In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

pastas = sorted(
    ROOT.joinpath("reproducao").glob("dados_SAM_EFT512_*")
)
assert pastas, "Nenhuma pasta de geração SAM encontrada."

print("Execuções encontradas:")
for p in pastas:
    print(" ", p.name)

DADOS_SAM_EFT = pastas[-1]
SCRIPT_GERAR_SAM = DADOS_SAM_EFT / "gerar_mascaras.py"

print("\nExecução selecionada:", DADOS_SAM_EFT)

concluido = DADOS_SAM_EFT / "concluido.json"
progresso = DADOS_SAM_EFT / "progresso.json"

if concluido.is_file():
    resumo = json.loads(concluido.read_text(encoding="utf-8"))
    print("\nGERAÇÃO CONCLUÍDA")
    print(json.dumps(
        {k: v for k, v in resumo.items()
         if k != "indices_sem_candidata"},
        indent=2, ensure_ascii=False,
    ))
else:
    print("\nAinda não há registro de conclusão.")
    if progresso.is_file():
        print(progresso.read_text(encoding="utf-8"))

logs = sorted(DADOS_SAM_EFT.glob("console_*.log"))
if logs:
    LOG_GERAR_SAM = logs[-1]
    print("\nÚLTIMAS LINHAS DO LOG:")
    print("\n".join(
        LOG_GERAR_SAM.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()[-15:]
    ))

In [ ]:
from datetime import datetime, timezone
import json

origem = ROOT / "exp/17_eft_tiny_512.ipynb"
assert origem.is_file(), origem

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
destino = origem.with_name(f"{origem.stem}_sem_saidas_{stamp}.ipynb")

with origem.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

for celula in notebook.get("cells", []):
    if celula.get("cell_type") == "code":
        celula["outputs"] = []
        celula["execution_count"] = None

# Remove eventual estado de widgets armazenado no notebook.
notebook.get("metadata", {}).pop("widgets", None)

with destino.open("x", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)

print(f"Original: {origem.stat().st_size / 1024**2:.1f} MiB")
print(f"Cópia:    {destino.stat().st_size / 1024**2:.1f} MiB")
print("Cópia criada:", destino)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys
import textwrap

import numpy as np

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

DADOS_SAM_EFT = ROOT / (
    "reproducao/dados_SAM_EFT512_20261021T111951917870Z"
)
REFERENCIA = ROOT / (
    "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
)
BASE = CODIGO / "treinar_ultima672_10000_wd001_cuda.py"

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def gravar(p, obj):
    with Path(p).open("x", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

assert (DADOS_SAM_EFT / "concluido.json").is_file()
resumo = ler(DADOS_SAM_EFT / "concluido.json")
assert resumo["imagens_processadas"] == 1304
assert resumo["imagens_elegiveis"] == 1231

cfg_ref = ler(REFERENCIA / "config.json")
for nome, esperado in cfg_ref["fontes_sha256"].items():
    assert sha(nome) == esperado, f"Fonte alterada: {nome}"

assert sha(REFERENCIA / "planos.json") == (
    "880b04375c4a25fa0efd326afc666011c47fe460c5180ad91653fe0cb4be024b"
)

referencia = ler(REFERENCIA / "planos.json")
indice_sam = ler(DADOS_SAM_EFT / "indice.json")
assert len(indice_sam) == 1304

por_indice = {r["indice"]: r for r in indice_sam}
assert len(por_indice) == 1304
elegiveis = {
    i for i, r in por_indice.items() if r["n_candidatas"] > 0
}
assert len(elegiveis) == 1231

ids_treino = {r["indice"] for r in referencia["treino"]}
ids_val = {r["indice"] for r in referencia["validacao"]}
assert set(por_indice) == ids_treino
assert not elegiveis & ids_val

# Confere as máscaras SAM disponíveis antes de criar os planos.
print("Conferindo caches SAM...")
for pos, i in enumerate(sorted(elegiveis), 1):
    r = por_indice[i]
    p = DADOS_SAM_EFT / r["cache"]
    assert sha(p) == r["cache_sha256"], p
    with np.load(p, allow_pickle=False) as d:
        assert d["candidatas"].shape == (
            r["n_candidatas"], 512, 512
        )
        assert d["candidatas"].reshape(r["n_candidatas"], -1).any(1).all()
    if pos % 300 == 0:
        print(f"Caches SAM: {pos}/{len(elegiveis)}")

# Mesma sequência para os dois braços.
filtrados = [
    r for r in referencia["treino"][:5000]
    if r["indice"] in elegiveis
]
assert {r["indice"] for r in filtrados} == elegiveis
agenda = [filtrados[i % len(filtrados)] for i in range(5000)]

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EXPERIMENTO_SAM = ROOT / "reproducao" / f"eft_SAM_vs_SP256_1231_{stamp}"
EXPERIMENTO_SAM.mkdir(parents=True, exist_ok=False)

rng = np.random.default_rng(1234)
ciclo_sam = []
ciclo_sp = []
dados_sp = Path(cfg_ref["dados"])

for antigo in agenda:
    s = por_indice[antigo["indice"]]
    assert s["imagem_sha256"] == antigo["imagem_sha256"]

    ciclo_sam.append({
        "indice": s["indice"],
        "nome": s["nome"],
        "imagem": s["imagem"],
        "imagem_sha256": s["imagem_sha256"],
        "split": "treino",
        "cache": str(DADOS_SAM_EFT / s["cache"]),
        "cache_sha256": s["cache_sha256"],
        # Aqui regiao indexa uma máscara independente, possivelmente sobreposta.
        "regiao": int(rng.integers(s["n_candidatas"])),
    })

    c = dict(antigo)
    c["cache"] = str(dados_sp / antigo["cache"])
    ciclo_sp.append(c)

planos = {
    "SAM": {
        "treino": ciclo_sam + [dict(r) for r in ciclo_sam],
        "validacao": [],
    },
    "SP256": {
        "treino": ciclo_sp + [dict(r) for r in ciclo_sp],
        "validacao": [],
    },
}
assert [
    r["indice"] for r in planos["SAM"]["treino"]
] == [
    r["indice"] for r in planos["SP256"]["treino"]
]

gravar(EXPERIMENTO_SAM / "comparacao.json", {
    "objetivo": "comparacao de pseudolabels no mesmo subconjunto",
    "referencia": str(REFERENCIA),
    "dados_SAM": str(DADOS_SAM_EFT),
    "dados_SAM_config_sha256": sha(DADOS_SAM_EFT / "config.json"),
    "indices_elegiveis": sorted(elegiveis),
    "indices_excluidos": sorted(ids_treino - elegiveis),
    "imagens": 1231,
    "episodios": 10000,
    "ordem": (
        "primeiros5000 historicos filtrados; repeticao ciclica ate5000; "
        "ciclo resultante repetido duas vezes"
    ),
    "episodios_historicos_apos_filtro": len(filtrados),
    "controle": "regioes SP256 historicas preservadas",
    "SAM": "sorteio uniforme por episodio, semente1234",
    "avaliacao": "separada, mesmas145 queries; penultima e SAM indice0",
})

nova_preparacao = '''
def preparar_plano():
    plano = ler(DADOS / "planos.json")
    assert len(plano["treino"]) == 10000
    assert plano["treino"][:5000] == plano["treino"][5000:]
    assert len({r["indice"] for r in plano["treino"]}) == 1231

    vistos = set()
    for r in plano["treino"]:
        assert r["split"] == "treino"
        cache = Path(r["cache"])
        assert cache.is_absolute()
        if str(cache) not in vistos:
            assert sha(cache) == r["cache_sha256"], cache
            vistos.add(str(cache))

    assert len(vistos) == 1231
    print("Plano conferido: 1231 imagens; 10000 episodios.", flush=True)
    return plano
'''

episodio_sam = '''
def episodio(registro):
    from PIL import Image

    with np.load(Path(registro["cache"]), allow_pickle=False) as d:
        mask512 = d["candidatas"][registro["regiao"]].copy()
    assert mask512.shape == (512, 512) and mask512.any()

    imagem_original = Path(registro["imagem"])
    assert sha(imagem_original) == registro["imagem_sha256"]
    with Image.open(imagem_original) as im:
        rgb = np.array(
            im.convert("RGB").resize(
                (672, 672), Image.Resampling.BILINEAR
            )
        )

    mask = np.array(
        Image.fromarray(mask512.astype(np.uint8)).resize(
            (672, 672), Image.Resampling.NEAREST
        )
    ) > 0
    assert mask.any()

    def imagem(a):
        t = torch.from_numpy(np.ascontiguousarray(a))
        t = t.permute(2, 0, 1)[None].to("cuda", dtype=torch.float32)
        media = t.new_tensor([123.675, 116.28, 103.53])[None, :, None, None]
        desvio = t.new_tensor([58.395, 57.12, 57.375])[None, :, None, None]
        return (t - media) / desvio

    return (
        imagem(rgb),
        torch.from_numpy(mask.copy())[None].to("cuda", dtype=torch.float32),
        imagem(np.fliplr(rgb)),
        torch.from_numpy(np.fliplr(mask).copy())[None].to(
            "cuda", dtype=torch.long
        ),
    )
'''

def trocar_funcao(codigo, nome, novo):
    nos = [
        n for n in ast.parse(codigo).body
        if isinstance(n, ast.FunctionDef) and n.name == nome
    ]
    assert len(nos) == 1, nome
    n = nos[0]
    linhas = codigo.splitlines(keepends=True)
    linhas[n.lineno - 1:n.end_lineno] = [
        textwrap.dedent(novo).strip() + "\n"
    ]
    return "".join(linhas)

def trocar_uma(codigo, antigo, novo):
    assert codigo.count(antigo) == 1, antigo
    return codigo.replace(antigo, novo, 1)

execucoes = []
for tipo in ("SAM", "SP256"):
    pasta_dados = EXPERIMENTO_SAM / f"plano_{tipo}"
    pasta_dados.mkdir()
    gravar(pasta_dados / "planos.json", planos[tipo])
    gravar(pasta_dados / "config.json", {
        "tipo_pseudolabel": tipo,
        "comparacao": str(EXPERIMENTO_SAM),
        "comparacao_sha256": sha(EXPERIMENTO_SAM / "comparacao.json"),
        "plano_sha256": sha(pasta_dados / "planos.json"),
        "imagens": 1231,
        "mascaras_manuais_utilizadas": False,
        "validacao_real": "separada; mesmas145 queries",
    })

    codigo = BASE.read_text(encoding="utf-8")
    codigo = trocar_funcao(codigo, "preparar_plano", nova_preparacao)

    codigo = trocar_uma(
        codigo,
        'DADOS = ROOT / "reproducao/'
        'eft_sp_dados512_regioes256_20261016T212513467126Z"',
        f"DADOS = Path({str(pasta_dados)!r})",
    )
    codigo = trocar_uma(
        codigo,
        '"experimento": "treino672 ultima; avaliacao penultima; 10k; WD 0.01"',
        f'"experimento": "EFT {tipo}, subconjunto1231, treino672, 10k",'
        f'\n        "pseudolabel": "{tipo}"',
    )
    codigo = trocar_uma(
        codigo,
        '"ordem": "historica, identica nos dois ciclos"',
        '"ordem": "historica filtrada e estendida; dois ciclos identicos"',
    )

    if tipo == "SAM":
        codigo = trocar_funcao(codigo, "episodio", episodio_sam)
        codigo = trocar_uma(
            codigo,
            '"mascara": "regiao256 recuperada do cache512 -> nearest 672"',
            '"mascara": "candidata SAM512 -> PIL nearest672"',
        )
        codigo = trocar_uma(
            codigo,
            '"superpixels_recalculados": False',
            '"superpixels_recalculados": False,\n'
            '            "mascaras_geradas_por_SAM": True',
        )

    codigo = trocar_uma(
        codigo,
        "TREINAMENTO CONCLUÍDO: 10000 passos em 672 pela ultima; WD=0.01.",
        f"TREINAMENTO CONCLUIDO: {tipo}, 1231 imagens, 10000 passos, 672.",
    )

    script = CODIGO / f"treinar_{tipo}_1231_{stamp}.py"
    compile(codigo, str(script), "exec")
    with script.open("x", encoding="utf-8") as f:
        f.write(codigo)

    run = EXPERIMENTO_SAM / f"treino_{tipo}"
    run.mkdir()
    execucoes.append({
        "tipo": tipo,
        "script": str(script),
        "script_sha256": sha(script),
        "run": str(run),
    })

gravar(EXPERIMENTO_SAM / "execucoes.json", execucoes)

# Coordenador: termina um treino antes de iniciar o seguinte.
coordenador = EXPERIMENTO_SAM / "executar_treinos.py"
coordenador.write_text(
    '''
from pathlib import Path
import hashlib
import json
import subprocess
import sys

pasta = Path(__file__).resolve().parent
execucoes = json.loads((pasta / "execucoes.json").read_text())
for e in execucoes:
    script = Path(e["script"])
    assert hashlib.sha256(script.read_bytes()).hexdigest() == e["script_sha256"]
    print("INICIANDO:", e["tipo"], flush=True)
    subprocess.run(
        [sys.executable, "-u", str(script), "--run", e["run"]],
        check=True,
    )
    print("CONCLUIDO:", e["tipo"], flush=True)
print("DOIS TREINOS CONCLUIDOS.", flush=True)
''',
    encoding="utf-8",
)

LOG_TREINOS_SAM = EXPERIMENTO_SAM / "console.log"
with LOG_TREINOS_SAM.open("x", encoding="utf-8") as log:
    processoTREINOS_SAM = subprocess.Popen(
        [sys.executable, "-u", str(coordenador)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID coordenador:", processoTREINOS_SAM.pid)
print("EXPERIMENTO — guarde:", EXPERIMENTO_SAM)
print("LOG:", LOG_TREINOS_SAM)
print("Ordem: SAM, depois controle SP256.")

In [ ]:
estado = processoTREINOS_SAM.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)
print("\n".join(
    LOG_TREINOS_SAM.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
CODIGO = ROOT / "exp/eft_tiny_512_v1"

assert "EXPERIMENTO_SAM" in globals(), (
    "Defina EXPERIMENTO_SAM com a pasta dos dois treinos."
)
EXPERIMENTO_SAM = Path(EXPERIMENTO_SAM)

BASE_AVALIADOR = CODIGO / "avaliar_penultima_treino672_10k_wd001.py"
assert BASE_AVALIADOR.is_file(), BASE_AVALIADOR

PASSOS_COMPARACAO = [
    0, 50, 100, 200, 500, 1000, 1500, 2000,
    3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000,
]

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

execucoes = ler(EXPERIMENTO_SAM / "execucoes.json")
por_tipo = {e["tipo"]: e for e in execucoes}
assert set(por_tipo) == {"SAM", "SP256"}

# Confere conclusão, checkpoints e sequência de imagens.
planos = {}
for tipo in ("SAM", "SP256"):
    run = Path(por_tipo[tipo]["run"])
    assert ler(run / "concluido.json")["passos"] == 10000
    planos[tipo] = ler(run / "planos.json")

    for passo in PASSOS_COMPARACAO:
        p = run / f"checkpoint_{passo:04d}.pt"
        assert p.is_file(), p

assert [
    r["indice"] for r in planos["SAM"]["treino"]
] == [
    r["indice"] for r in planos["SP256"]["treino"]
]

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AVAL_SAM_SP = EXPERIMENTO_SAM / f"avaliacao_S00_{stamp}"
AVAL_SAM_SP.mkdir(parents=True, exist_ok=False)

tarefas = []
for tipo in ("SAM", "SP256"):
    run = Path(por_tipo[tipo]["run"])
    codigo = BASE_AVALIADOR.read_text(encoding="utf-8")
    arvore = ast.parse(codigo)

    alteracoes = []
    for nome, expressao in {
        "TREINO": f"Path({str(run)!r})",
        "PASSOS": repr(PASSOS_COMPARACAO),
    }.items():
        encontrados = [
            n for n in arvore.body
            if isinstance(n, ast.Assign)
            and any(
                isinstance(t, ast.Name) and t.id == nome
                for t in n.targets
            )
        ]
        assert len(encontrados) == 1, nome
        n = encontrados[0]
        alteracoes.append((
            n.lineno, n.end_lineno, f"{nome} = {expressao}\n"
        ))

    linhas = codigo.splitlines(keepends=True)
    for inicio, fim, substituto in sorted(alteracoes, reverse=True):
        linhas[inicio - 1:fim] = [substituto]
    codigo = "".join(linhas)

    script = CODIGO / f"avaliar_{tipo}_1231_{stamp}.py"
    compile(codigo, str(script), "exec")
    with script.open("x", encoding="utf-8") as f:
        f.write(codigo)

    tarefas.append({
        "tipo": tipo,
        "treino": str(run),
        "treino_config_sha256": sha(run / "config.json"),
        "script": str(script),
        "script_sha256": sha(script),
        "out": str(AVAL_SAM_SP / tipo),
    })

manifesto = {
    "avaliador_base": str(BASE_AVALIADOR),
    "avaliador_base_sha256": sha(BASE_AVALIADOR),
    "alteracoes": ["TREINO", "PASSOS"],
    "passos": PASSOS_COMPARACAO,
    "tarefas": tarefas,
    "limite": "validacao exploratoria S00; nao e teste",
}
(AVAL_SAM_SP / "lancamento.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

coordenador = AVAL_SAM_SP / "executar_avaliacoes.py"
coordenador.write_text(
    '''
from pathlib import Path
import hashlib
import json
import subprocess
import sys

pasta = Path(__file__).resolve().parent
manifesto = json.loads((pasta / "lancamento.json").read_text())

for tarefa in manifesto["tarefas"]:
    script = Path(tarefa["script"])
    assert hashlib.sha256(script.read_bytes()).hexdigest() == tarefa["script_sha256"]
    print("INICIANDO AVALIACAO:", tarefa["tipo"], flush=True)
    subprocess.run(
        [sys.executable, "-u", str(script), "--out", tarefa["out"]],
        check=True,
    )
    print("AVALIACAO FINALIZADA:", tarefa["tipo"], flush=True)

(pasta / "concluido.json").write_text(
    json.dumps({"avaliacoes_concluidas": ["SAM", "SP256"]}, indent=2)
)
print("DUAS AVALIACOES CONCLUIDAS.", flush=True)
''',
    encoding="utf-8",
)

LOG_AVAL_SAM_SP = AVAL_SAM_SP / "console.log"
with LOG_AVAL_SAM_SP.open("x", encoding="utf-8") as log:
    processoAVAL_SAM_SP = subprocess.Popen(
        [sys.executable, "-u", str(coordenador)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID:", processoAVAL_SAM_SP.pid)
print("AVALIAÇÃO — guarde:", AVAL_SAM_SP)
print("LOG:", LOG_AVAL_SAM_SP)
print("Ordem: treino SAM, depois controle SP256.")

In [ ]:
estado = processoAVAL_SAM_SP.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)
print("\n".join(
    LOG_AVAL_SAM_SP.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-20:]
))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXPERIMENTO_SAM = ROOT / (
    "reproducao/eft_SAM_vs_SP256_1231_20261021T194500705194Z"
)
AVAL_ANTIGA = EXPERIMENTO_SAM / "avaliacao_S00_20261021T203506850646Z"

REFERENCIA_DIVISAO = ROOT / (
    "reproducao/eft_ultima672_10000_wd001_20261018T141350233303Z"
)
HASH_PLANO_REF = (
    "880b04375c4a25fa0efd326afc666011c47fe460c5180ad91653fe0cb4be024b"
)

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

plano_ref_path = REFERENCIA_DIVISAO / "planos.json"
assert sha(plano_ref_path) == HASH_PLANO_REF
plano_ref = ler(plano_ref_path)
val_ids = {r["indice"] for r in plano_ref["validacao"]}
assert len(plano_ref["validacao"]) == len(val_ids) == 145

manifesto_antigo = ler(AVAL_ANTIGA / "lancamento.json")
assert {t["tipo"] for t in manifesto_antigo["tarefas"]} == {"SAM", "SP256"}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AVAL_SAM_SP = EXPERIMENTO_SAM / f"avaliacao_S00_corrigida_{stamp}"
AVAL_SAM_SP.mkdir(parents=True, exist_ok=False)

marcador = '    planos = ler(TREINO / "planos.json")'

correcao = marcador + f'''
    # Recupera somente os registros de validacao da divisao original.
    # O plano de treino e os checkpoints permanecem intactos.
    origem_divisao = Path({str(plano_ref_path)!r})
    assert sha(origem_divisao) == {HASH_PLANO_REF!r}
    plano_divisao = ler(origem_divisao)
    assert planos["validacao"] == [], "Plano diferente do esperado."
    assert len(plano_divisao["validacao"]) == 145
    assert len({{r["indice"] for r in plano_divisao["validacao"]}}) == 145
    assert not (
        {{r["indice"] for r in planos["treino"]}}
        & {{r["indice"] for r in plano_divisao["validacao"]}}
    )
    planos = dict(planos, validacao=plano_divisao["validacao"])
'''

tarefas = []

for tarefa in manifesto_antigo["tarefas"]:
    treino = Path(tarefa["treino"])
    assert sha(treino / "config.json") == tarefa["treino_config_sha256"]
    assert ler(treino / "concluido.json")["passos"] == 10000

    plano_treino = ler(treino / "planos.json")
    assert plano_treino["validacao"] == []
    assert not (
        {r["indice"] for r in plano_treino["treino"]} & val_ids
    )

    original = Path(tarefa["script"])
    assert sha(original) == tarefa["script_sha256"]
    codigo = original.read_text(encoding="utf-8")
    assert codigo.count(marcador) == 1, (
        "Trecho de carregamento do plano diferente do esperado."
    )
    codigo = codigo.replace(marcador, correcao, 1)

    script = original.with_name(
        f"avaliar_{tarefa['tipo']}_1231_divisao_corrigida_{stamp}.py"
    )
    compile(codigo, str(script), "exec")
    with script.open("x", encoding="utf-8") as f:
        f.write(codigo)

    nova = dict(tarefa)
    nova.update({
        "script": str(script),
        "script_sha256": sha(script),
        "script_anterior": str(original),
        "out": str(AVAL_SAM_SP / tarefa["tipo"]),
    })
    tarefas.append(nova)

manifesto = dict(manifesto_antigo)
manifesto.update({
    "tarefas": tarefas,
    "avaliacao_anterior": str(AVAL_ANTIGA),
    "correcao": "validacao recuperada da divisao original apenas em memoria",
    "plano_divisao": str(plano_ref_path),
    "plano_divisao_sha256": HASH_PLANO_REF,
})

(AVAL_SAM_SP / "lancamento.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Reutiliza o coordenador: ele lê o manifesto da própria pasta.
coordenador = AVAL_SAM_SP / "executar_avaliacoes.py"
codigo_coordenador = (
    AVAL_ANTIGA / "executar_avaliacoes.py"
).read_text(encoding="utf-8")
compile(codigo_coordenador, str(coordenador), "exec")
coordenador.write_text(codigo_coordenador, encoding="utf-8")

LOG_AVAL_SAM_SP = AVAL_SAM_SP / "console.log"
with LOG_AVAL_SAM_SP.open("x", encoding="utf-8") as log:
    processoAVAL_SAM_SP = subprocess.Popen(
        [sys.executable, "-u", str(coordenador)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

PASSOS_COMPARACAO = manifesto["passos"]

print("PID:", processoAVAL_SAM_SP.pid)
print("AVALIAÇÃO CORRIGIDA:", AVAL_SAM_SP)
print("LOG:", LOG_AVAL_SAM_SP)
print("Validação: mesmas 145 queries da divisão original.")

In [ ]:
from pathlib import Path
import json
import math

assert "AVAL_SAM_SP" in globals(), (
    "A variável AVAL_SAM_SP não está definida. "
    "Defina-a com o caminho da pasta avaliacao_S00_corrigida desta execução."
)
pasta = Path(AVAL_SAM_SP)

def ler_json(caminho):
    return json.loads(Path(caminho).read_text(encoding="utf-8"))

manifesto = ler_json(pasta / "lancamento.json")
assert (pasta / "concluido.json").is_file(), (
    f"As duas avaliações ainda não estão registradas como concluídas: {pasta}"
)

passos = sorted(int(p) for p in manifesto["passos"])
tarefas = {t["tipo"]: t for t in manifesto["tarefas"]}
assert {"SAM", "SP256"}.issubset(tarefas)

resultados = {}
for tipo in ("SAM", "SP256"):
    saida = Path(tarefas[tipo]["out"])
    resultados[tipo] = {}

    for passo in passos:
        arquivo = saida / f"passo_{passo:04d}" / "resumo.json"
        assert arquivo.is_file(), f"Resumo ausente: {arquivo}"
        resumo = ler_json(arquivo)

        assert int(resumo["passo"]) == passo, arquivo
        assert int(resumo["n"]) == 145, arquivo
        assert all(
            math.isfinite(float(resumo[chave]))
            for chave in ("ALP_Dice", "SAM_Dice")
        ), arquivo

        resultados[tipo][passo] = resumo

# Os dois modelos devem começar com o mesmo resultado.
for metrica in ("ALP_Dice", "SAM_Dice"):
    diferenca = abs(
        resultados["SAM"][0][metrica] - resultados["SP256"][0][metrica]
    )
    assert diferenca < 1e-6, (
        f"Resultados iniciais diferentes em {metrica}: {diferenca}. "
        "Confira os pesos iniciais e as configurações de avaliação."
    )

print("Pseudorrótulos de treino: SAM versus SP256")
print("Avaliação: S00, mesmas 145 queries")
print()
print(
    "Passo | ALP SP256 | ALP SAM   | Final SP256 | Final SAM | Δ final (p.p.)"
)

linhas = []
for passo in passos:
    controle = resultados["SP256"][passo]
    candidato = resultados["SAM"][passo]
    delta = 100 * (candidato["SAM_Dice"] - controle["SAM_Dice"])

    print(
        f"{passo:5d} | "
        f"{controle['ALP_Dice']:.6f}  | "
        f"{candidato['ALP_Dice']:.6f}  | "
        f"{controle['SAM_Dice']:.6f}    | "
        f"{candidato['SAM_Dice']:.6f}  | "
        f"{delta:+.2f}"
    )

    linhas.append({
        "passo": passo,
        "SP256": controle,
        "SAM": candidato,
        "delta_final_SAM_menos_SP256_pp": delta,
    })

# Critério: maior Dice ALP+SAM; empate favorece menor passo.
melhores = {
    tipo: max(
        resultados[tipo].values(),
        key=lambda r: (r["SAM_Dice"], -int(r["passo"])),
    )
    for tipo in ("SAM", "SP256")
}

delta_melhores = 100 * (
    melhores["SAM"]["SAM_Dice"] - melhores["SP256"]["SAM_Dice"]
)

print("\n=== MELHOR CHECKPOINT POR TREINO ===")
for tipo in ("SP256", "SAM"):
    melhor = melhores[tipo]
    print(
        f"Pseudorrótulos {tipo}: passo {melhor['passo']} | "
        f"Dice ALP={100 * melhor['ALP_Dice']:.2f}% | "
        f"Dice ALP+SAM={100 * melhor['SAM_Dice']:.2f}%"
    )

print(f"\nDiferença entre os melhores (SAM − SP256): {delta_melhores:+.2f} p.p.")
print("Final = ALP + refinamento SAM, em ambos os treinos.")
print("Seleção exploratória na validação S00; não é resultado de teste.")

comparacao = {
    "criterio": "maior Dice medio ALP+SAM; empate favorece menor passo",
    "por_checkpoint": linhas,
    "melhores": melhores,
    "delta_melhores_pp": delta_melhores,
    "limite": (
        "Validacao exploratoria S00 com 145 queries; "
        "nao e teste nem mede variacao entre treinamentos."
    ),
}

arquivo = pasta / "comparacao.json"
temporario = arquivo.with_suffix(".json.tmp")
temporario.write_text(
    json.dumps(comparacao, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
temporario.replace(arquivo)
print("\nComparação salva em:", arquivo)

In [ ]:
from pathlib import Path
import json

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")

ANTERIOR = ROOT / (
    "reproducao/"
    "multissuporte_candidatos_AB_20261019T000102056755Z"
)

ATUAL = ROOT / (
    "reproducao/"
    "eft_SAM_vs_SP256_1231_20261021T194500705194Z/"
    "avaliacao_S00_corrigida_20261021T203734019017Z"
)

assert ANTERIOR.is_dir(), ANTERIOR
assert ATUAL.is_dir(), ATUAL

print("=== MANIFESTO DA AVALIAÇÃO ATUAL ===")
manifesto = json.loads(
    (ATUAL / "lancamento.json").read_text(encoding="utf-8")
)
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

print("\n=== SCRIPTS DO MULTISSUPORTE ANTERIOR ===")
scripts = sorted(ANTERIOR.rglob("*.py"))
for p in scripts:
    print(p.relative_to(ANTERIOR), "|", p.stat().st_size, "bytes")

# Coordenadores e scripts principais na raiz.
principais = [p for p in scripts if p.parent == ANTERIOR]

# Um avaliador do candidato A:
# treino672, penúltima, passo500, SAM[0].
avaliadores_a = [
    p for p in scripts
    if p not in principais
    and any(
        parte.startswith("A_")
        for parte in p.relative_to(ANTERIOR).parts
    )
]

selecionados = principais + avaliadores_a[:1]

for p in selecionados:
    print(f"\n=== CÓDIGO: {p.relative_to(ANTERIOR)} ===")
    print(p.read_text(encoding="utf-8"))

if not selecionados:
    print(
        "\nNenhum script principal localizado nesta pasta. "
        "Envie a listagem acima para localizarmos o avaliador."
    )

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
ANTERIOR = ROOT / "reproducao/multissuporte_candidatos_AB_20261019T000102056755Z"
EXPERIMENTO = ROOT / "reproducao/eft_SAM_vs_SP256_1231_20261021T194500705194Z"
S00 = EXPERIMENTO / "avaliacao_S00_corrigida_20261021T203734019017Z"

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def gravar(p, objeto):
    Path(p).write_text(
        json.dumps(objeto, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

def substituir_unico(codigo, antigo, novo):
    assert codigo.count(antigo) == 1, (
        f"Esperada uma ocorrência; encontradas {codigo.count(antigo)}: {antigo}"
    )
    return codigo.replace(antigo, novo, 1)

def alterar_constante(codigo, nome, expressao):
    arvore = ast.parse(codigo)
    encontrados = [
        n for n in arvore.body
        if isinstance(n, ast.Assign)
        and any(isinstance(t, ast.Name) and t.id == nome for t in n.targets)
    ]
    assert len(encontrados) == 1, nome
    n = encontrados[0]
    linhas = codigo.splitlines(keepends=True)
    linhas[n.lineno - 1:n.end_lineno] = [f"{nome} = {expressao}\n"]
    return "".join(linhas)

# Evita iniciar duas cópias nesta sessão.
anterior = globals().get("processoMULTI_SAM_SP")
assert anterior is None or anterior.poll() is not None, (
    "A avaliação multissuporte já está executando."
)

assert (S00 / "concluido.json").is_file()
manifesto = ler(S00 / "lancamento.json")
tarefas = {t["tipo"]: t for t in manifesto["tarefas"]}

BASE = ANTERIOR / "A_672_penultima_500_indice0.py"
COPIA_BASE = (
    ANTERIOR / "A_672_penultima_500_indice0/S01/fontes/"
    "00_A_672_penultima_500_indice0.py"
)
EXECUTOR_BASE = ANTERIOR / "executar_AB.py"

assert sha(BASE) == sha(COPIA_BASE), "O avaliador anterior mudou."
codigo_base = BASE.read_text(encoding="utf-8")
executor_base = EXECUTOR_BASE.read_text(encoding="utf-8")

DIVISAO = Path(manifesto["plano_divisao"])
HASH_DIVISAO = manifesto["plano_divisao_sha256"]
assert sha(DIVISAO) == HASH_DIVISAO
divisao = ler(DIVISAO)
assert len(divisao["validacao"]) == 145

PLANO_SUP = ROOT / "reproducao/multissuporte_tiny_eft2000_kvasir_v1/plano.json"
plano_sup = ler(PLANO_SUP)
assert [s["id"] for s in plano_sup["suportes"]] == [
    f"S{i:02d}" for i in range(20)
]

def caminho_atual(p):
    p = Path(p)
    if p.is_relative_to("/workspace"):
        p = ROOT.parent / p.relative_to("/workspace")
    return str(p.resolve())

candidatos = [
    ("A_SP256_500", "SP256", 500),
    ("B_SAM_4000", "SAM", 4000),
]

# Verifica os dois treinos antes de criar/iniciar a avaliação.
imagens_treino = {}
for nome, tipo, passo in candidatos:
    t = tarefas[tipo]
    treino = Path(t["treino"])
    assert sha(treino / "config.json") == t["treino_config_sha256"]
    assert ler(treino / "concluido.json")["passos"] == 10000
    assert (treino / f"checkpoint_{passo:04d}.pt").is_file()

    plano = ler(treino / "planos.json")
    assert plano["validacao"] == []
    ids = {r["indice"] for r in plano["treino"]}
    assert len(ids) == 1231
    assert not ids & {r["indice"] for r in divisao["validacao"]}
    imagens_treino[tipo] = {
        caminho_atual(r["imagem"]) for r in plano["treino"]
    }

assert imagens_treino["SAM"] == imagens_treino["SP256"]

pool_original = {
    caminho_atual(r["imagem"]) for r in divisao["treino"]
}
auditoria_suportes = []
for s in plano_sup["suportes"][1:]:
    imagem = caminho_atual(s["meta"]["imagem"])
    assert imagem in pool_original
    auditoria_suportes.append({
        "suporte": s["id"],
        "imagem": imagem,
        "presente_nos_dois_treinos_EFT": imagem in imagens_treino["SAM"],
    })

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
MULTI_SAM_SP = EXPERIMENTO / f"multissuporte_SAM_vs_SP256_{stamp}"
MULTI_SAM_SP.mkdir(parents=True, exist_ok=False)

# Recupera apenas a validação; mantém o plano real de treino.
insercao_divisao = f'''
    plano_divisao_path = Path({str(DIVISAO)!r})
    assert sha(plano_divisao_path) == {HASH_DIVISAO!r}
    plano_divisao = ler(plano_divisao_path)
    assert planos["validacao"] == []
    assert len(plano_divisao["validacao"]) == 145
    assert not (
        {{r["indice"] for r in planos["treino"]}}
        & {{r["indice"] for r in plano_divisao["validacao"]}}
    )
    planos = dict(planos, validacao=plano_divisao["validacao"])
'''

registros = []
for nome, tipo, passo in candidatos:
    treino = Path(tarefas[tipo]["treino"])
    codigo = alterar_constante(codigo_base, "TREINO", f"Path({str(treino)!r})")
    codigo = alterar_constante(codigo, "PASSOS", repr([passo]))

    marcador = '    planos = ler(TREINO / "planos.json")'
    codigo = substituir_unico(codigo, marcador, marcador + insercao_divisao)

    # Os suportes continuam pertencendo ao pool original de treino.
    # Registra separadamente se participaram do EFT com 1231 imagens.
    codigo = substituir_unico(
        codigo,
        'str(atual(r["imagem"])) for r in planos["treino"]',
        'str(atual(r["imagem"])) for r in plano_divisao["treino"]',
    )
    codigo = codigo.replace(
        "Suporte sorteado fora do conjunto de treino registrado.",
        "Suporte fora do pool original de treino.",
    )

    codigo = substituir_unico(
        codigo, '"passo": 500,', f'"passo": {passo},'
    )
    codigo = substituir_unico(
        codigo,
        '"passo_0500/resumo.json"',
        f'"passo_{passo:04d}/resumo.json"',
    )
    codigo = codigo.replace("A_672_penultima_500_indice0", nome)

    marcador = (
        '    config["finalidade"] = '
        '"comparacao multissuporte dos candidatos A e B"'
    )
    codigo = substituir_unico(
        codigo,
        marcador,
        f'''    config["finalidade"] = "SAM versus SP256; mesmos suportes S01-S19"
    config["pseudorrotulos_treino"] = {tipo!r}
    config["plano_divisao_sha256"] = sha(plano_divisao_path)
    config["suporte_presente_no_treino_EFT"] = sup_meta["imagem"] in {{
        str(atual(r["imagem"])) for r in planos["treino"]
    }}''',
    )

    script = MULTI_SAM_SP / f"{nome}.py"
    compile(codigo, str(script), "exec")
    script.write_text(codigo, encoding="utf-8")
    registros.append({
        "nome": nome,
        "tipo": tipo,
        "passo": passo,
        "treino": str(treino),
        "treino_config_sha256": sha(treino / "config.json"),
        "checkpoint_sha256": sha(treino / f"checkpoint_{passo:04d}.pt"),
        "script": str(script),
        "script_sha256": sha(script),
    })

nomes = [r["nome"] for r in registros]
executor = alterar_constante(
    executor_base, "OUT", f"Path({str(MULTI_SAM_SP)!r})"
)
executor = alterar_constante(executor, "NOMES", repr(nomes))

# Amplia as verificações de igualdade do protocolo entre A e B.
executor = substituir_unico(
    executor,
    '            "plano_suportes_sha256",',
    '''            "plano_suportes_sha256",
            "extracao", "selecao_sam", "alp_inferencia",
            "plano_divisao_sha256", "suporte_presente_no_treino_EFT",''',
)

script_executor = MULTI_SAM_SP / "executar_comparacao.py"
compile(executor, str(script_executor), "exec")
script_executor.write_text(executor, encoding="utf-8")

gravar(MULTI_SAM_SP / "lancamento.json", {
    "candidatos": registros,
    "avaliador_base": str(BASE),
    "avaliador_base_sha256": sha(BASE),
    "executor_base_sha256": sha(EXECUTOR_BASE),
    "executor_sha256": sha(script_executor),
    "plano_suportes_sha256": sha(PLANO_SUP),
    "plano_divisao_sha256": HASH_DIVISAO,
    "auditoria_suportes": auditoria_suportes,
    "A": nomes[0],
    "B": nomes[1],
    "limite": "validacao S01-S19; variacao entre suportes, nao entre treinos",
})

LOG_MULTI_SAM_SP = MULTI_SAM_SP / "console.log"
with LOG_MULTI_SAM_SP.open("w", encoding="utf-8") as log:
    processoMULTI_SAM_SP = subprocess.Popen(
        [sys.executable, "-u", str(script_executor)],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

gravar(MULTI_SAM_SP / "processo.json", {
    "pid": processoMULTI_SAM_SP.pid,
    "python": sys.executable,
})

print("PID:", processoMULTI_SAM_SP.pid)
print("EXECUÇÃO — guarde este caminho:", MULTI_SAM_SP)
print("A: SP256, passo 500")
print("B: pseudomáscaras SAM, passo 4000")
print("19 suportes × 145 queries × 2 candidatos = 5510 avaliações")
print(
    "Suportes presentes nos dois treinos EFT:",
    sum(s["presente_nos_dois_treinos_EFT"] for s in auditoria_suportes),
    "/ 19",
)
print("LOG:", LOG_MULTI_SAM_SP)

In [ ]:
estado = processoMULTI_SAM_SP.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)

linhas = LOG_MULTI_SAM_SP.read_text(
    encoding="utf-8", errors="replace"
).splitlines()
print("\n".join(linhas[-25:]))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast
import hashlib
import json
import subprocess
import sys

ROOT = Path("/home/1542521/cassio_pereira/ProtoSAM-repro")
EXPERIMENTO = ROOT / "reproducao/eft_SAM_vs_SP256_1231_20261021T194500705194Z"

def ler(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

def sha(p):
    h = hashlib.sha256()
    with Path(p).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

# Recupera a comparação concluída, inclusive após reinício do kernel.
if "MULTI_SAM_SP" in globals():
    ORIGEM_CAMADAS = Path(MULTI_SAM_SP)
else:
    candidatas = [
        p for p in EXPERIMENTO.glob("multissuporte_SAM_vs_SP256_*")
        if (p / "concluido.json").is_file()
    ]
    assert len(candidatas) == 1, (
        "Defina MULTI_SAM_SP com o caminho da comparação desejada. "
        f"Execuções concluídas encontradas: {candidatas}"
    )
    ORIGEM_CAMADAS = candidatas[0]

assert (ORIGEM_CAMADAS / "concluido.json").is_file()

anterior = globals().get("processoULTIMA_SAM")
assert anterior is None or anterior.poll() is not None, (
    "A avaliação da última camada já está executando."
)

manifesto = ler(ORIGEM_CAMADAS / "lancamento.json")
candidato = [
    r for r in manifesto["candidatos"]
    if r["tipo"] == "SAM" and r["passo"] == 4000
]
assert len(candidato) == 1
candidato = candidato[0]

base = Path(candidato["script"])
treino = Path(candidato["treino"])
assert sha(base) == candidato["script_sha256"]
assert sha(treino / "config.json") == candidato["treino_config_sha256"]
assert sha(treino / "checkpoint_4000.pt") == candidato["checkpoint_sha256"]

PENULTIMA_SAM = ORIGEM_CAMADAS / candidato["nome"]
assert ler(PENULTIMA_SAM / "concluido.json")["n_suportes"] == 19

codigo = base.read_text(encoding="utf-8")

# O avaliador constrói e carrega primeiro o modelo da última camada.
# Retirar esta chamada preserva a extração original e sua norm aprendida.
marcador = "            instalar_penultima(modelo)"
assert codigo.count(marcador) == 1
codigo = codigo.replace(
    marcador,
    '            print("Extração: última camada + norm aprendida", flush=True)',
    1,
)

# Atualiza o registro efetivo de extração no config.json.
arvore = ast.parse(codigo)
atribuicoes = [
    n for n in ast.walk(arvore)
    if isinstance(n, ast.Assign)
    and any(
        isinstance(t, ast.Subscript)
        and isinstance(t.value, ast.Name)
        and t.value.id == "config"
        and isinstance(t.slice, ast.Constant)
        and t.slice.value == "extracao"
        for t in n.targets
    )
]
assert len(atribuicoes) == 1
n = atribuicoes[0]

extracao = {
    "estagio_indice": 3,
    "canais": 768,
    "grade_nativa": [21, 21],
    "normalizacao_saida": "outnorm3 aprendida",
    "novo_treinamento": False,
}
linhas = codigo.splitlines(keepends=True)
linhas[n.lineno - 1:n.end_lineno] = [
    " " * n.col_offset + f'config["extracao"] = {extracao!r}\n'
]
codigo = "".join(linhas)

nome_novo = "SAM_4000_ultima_indice0"
codigo = codigo.replace(candidato["nome"], nome_novo)

# Confirma que checkpoint e seleção SAM permanecem iguais.
arvore = ast.parse(codigo)
passos = [
    ast.literal_eval(n.value)
    for n in arvore.body
    if isinstance(n, ast.Assign)
    and any(isinstance(t, ast.Name) and t.id == "PASSOS" for t in n.targets)
]
assert passos == [[4000]]
assert "instalar_selecao_sam(protosam, 'indice0')" in codigo

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CAMADAS_SAM = EXPERIMENTO / f"multissuporte_camadas_SAM4000_{stamp}"
CAMADAS_SAM.mkdir(parents=True, exist_ok=False)

script = CAMADAS_SAM / "avaliar_ultima.py"
compile(codigo, str(script), "exec")
script.write_text(codigo, encoding="utf-8")

ULTIMA_SAM = CAMADAS_SAM / nome_novo
registro = {
    "penultima_resultados": str(PENULTIMA_SAM),
    "ultima_resultados": str(ULTIMA_SAM),
    "checkpoint": str(treino / "checkpoint_4000.pt"),
    "checkpoint_sha256": candidato["checkpoint_sha256"],
    "script_base": str(base),
    "script_base_sha256": sha(base),
    "script_ultima": str(script),
    "script_ultima_sha256": sha(script),
    "alteracao": "extracao ultima versus penultima; mesmo checkpoint e SAM indice0",
    "limite": "validacao S01-S19; nao e teste",
}
(CAMADAS_SAM / "lancamento.json").write_text(
    json.dumps(registro, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

LOG_ULTIMA_SAM = CAMADAS_SAM / "console.log"
with LOG_ULTIMA_SAM.open("w", encoding="utf-8") as log:
    processoULTIMA_SAM = subprocess.Popen(
        [
            sys.executable, "-u", str(script),
            "--out", str(ULTIMA_SAM),
        ],
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("PID:", processoULTIMA_SAM.pid)
print("EXECUÇÃO — guarde este caminho:", CAMADAS_SAM)
print("Checkpoint: SAM 4000 | seleção SAM: índice 0")
print("Última: 19 suportes × 145 queries = 2755 avaliações")
print("Penúltima: resultados anteriores reutilizados")
print("LOG:", LOG_ULTIMA_SAM)

In [ ]:
estado = processoULTIMA_SAM.poll()
print(
    "Estado:",
    "executando" if estado is None else f"encerrado, código {estado}",
)
print("\n".join(
    LOG_ULTIMA_SAM.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()[-25:]
))

In [ ]:
import csv
import json
import numpy as np

assert processoULTIMA_SAM.poll() == 0, (
    "A avaliação ainda não terminou com sucesso. Confira o log."
)

pen = {
    r["suporte"]: r
    for r in ler(PENULTIMA_SAM / "por_suporte.json")
}
ult = {
    r["suporte"]: r
    for r in ler(ULTIMA_SAM / "por_suporte.json")
}
assert set(pen) == set(ult) == {f"S{i:02d}" for i in range(1, 20)}

linhas = []
for sid in sorted(pen):
    cp = ler(PENULTIMA_SAM / sid / "config.json")
    cu = ler(ULTIMA_SAM / sid / "config.json")

    for chave in (
        "treino", "treino_config_sha256", "passos",
        "checkpoints_sha256", "episodios", "suporte",
        "suporte_tensor_sha256", "encoder", "grade_alp",
        "canvas", "sam", "sam_sha256", "selecao_sam",
        "alp_inferencia", "plano_suportes_sha256",
        "plano_divisao_sha256", "suporte_presente_no_treino_EFT",
    ):
        assert cp[chave] == cu[chave], (sid, chave)

    assert cp["extracao"]["estagio_indice"] == 2
    assert cu["extracao"]["estagio_indice"] == 3
    assert pen[sid]["n"] == ult[sid]["n"] == 145
    assert pen[sid]["passo"] == ult[sid]["passo"] == 4000

    linhas.append({
        "suporte": sid,
        "ALP_penultima": pen[sid]["ALP_Dice"],
        "ALP_ultima": ult[sid]["ALP_Dice"],
        "SAM_penultima": pen[sid]["SAM_Dice"],
        "SAM_ultima": ult[sid]["SAM_Dice"],
        "delta_ALP_pp": 100 * (
            ult[sid]["ALP_Dice"] - pen[sid]["ALP_Dice"]
        ),
        "delta_SAM_pp": 100 * (
            ult[sid]["SAM_Dice"] - pen[sid]["SAM_Dice"]
        ),
    })

print("Suporte | ALP pen. | ALP últ. | SAM pen. | SAM últ. | Δ SAM p.p.")
for r in linhas:
    print(
        f"{r['suporte']:>7} | "
        f"{r['ALP_penultima']:.6f} | {r['ALP_ultima']:.6f} | "
        f"{r['SAM_penultima']:.6f} | {r['SAM_ultima']:.6f} | "
        f"{r['delta_SAM_pp']:+.2f}"
    )

resumo = {
    "checkpoint": 4000,
    "n_suportes": 19,
    "queries_por_suporte": 145,
    "selecao_sam": "indice0",
    "limite": "validacao; variacao entre suportes, nao entre treinamentos",
}

print("\n=== MÉDIA ± DP ENTRE SUPORTES ===")
for etapa in ("ALP", "SAM"):
    resumo[etapa] = {}
    for camada in ("penultima", "ultima"):
        valores = np.array([r[f"{etapa}_{camada}"] for r in linhas])
        resumo[etapa][camada] = {
            "media": float(valores.mean()),
            "dp_ddof1": float(valores.std(ddof=1)),
        }
        print(
            f"{etapa} {camada}: "
            f"{100*valores.mean():.2f}% ± "
            f"{100*valores.std(ddof=1):.2f} p.p."
        )

    deltas = np.array([r[f"delta_{etapa}_pp"] for r in linhas])
    resumo[etapa]["comparacao"] = {
        "delta_medio_ultima_menos_penultima_pp": float(deltas.mean()),
        "ultima_maior": int((deltas > 0).sum()),
        "penultima_maior": int((deltas < 0).sum()),
        "empates": int((deltas == 0).sum()),
    }
    print("Comparação:", resumo[etapa]["comparacao"])

with (CAMADAS_SAM / "comparacao_pareada.csv").open(
    "w", encoding="utf-8", newline=""
) as f:
    w = csv.DictWriter(f, fieldnames=list(linhas[0]))
    w.writeheader()
    w.writerows(linhas)

(CAMADAS_SAM / "comparacao.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nComparação salva em:", CAMADAS_SAM)